## Ablation Study of Model Components


In [ ]:
import glob
import json
import math
import os
import pickle
import sys

source_folder = "/beegfs/halder/GITHUB/RESEARCH/crop-yield-forecasting-germany/src"
sys.path.append(source_folder)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import pearsonr
from sklearn.metrics import (
    mean_absolute_error,
    mean_absolute_percentage_error,
    mean_squared_error,
    r2_score,
)

plt.rcParams["font.family"] = "DeJavu Serif"
plt.rcParams["font.serif"] = "Times New Roman"

import warnings

warnings.filterwarnings("ignore")

out_dir = "/beegfs/halder/GITHUB/RESEARCH/crop-yield-forecasting-germany/output/figures"
ablation_base = (
    "/beegfs/halder/GITHUB/RESEARCH/crop-yield-forecasting-germany/src/train/ablation"
)

## Load ablation results and compute metrics


In [ ]:
# --- Configuration ---
crops = ["winter_wheat", "winter_barley", "silage_maize"]

crop_labels = {
    "winter_wheat": "Winter Wheat",
    "winter_barley": "Winter Barley",
    "silage_maize": "Silage Maize",
}

scenarios = [
    "full_model",
    "no_vsn",
    "no_temporal_conv",
    "no_lstm",
    "no_attention",
    "no_static_enrichment",
    "no_pyramidal_pooling",
]

scenario_labels = {
    "full_model": "Full Model",
    "no_vsn": "w/o VSN",
    "no_temporal_conv": "w/o Temporal Conv",
    "no_lstm": "w/o LSTM",
    "no_attention": "w/o Attention",
    "no_static_enrichment": "w/o Static Enrichment",
    "no_pyramidal_pooling": "w/o Temporal Pooling",
}

# Load scalers for de-normalization
scalers = {}
for crop in crops:
    scaler_path = os.path.join(source_folder, "scaler", f"scaler_{crop}.json")
    with open(scaler_path, "r") as f:
        scalers[crop] = json.load(f)

# --- Load predictions & compute metrics (validation + test combined) ---
rows = []

for crop in crops:
    target_mean = scalers[crop]["yield_mean"]
    target_std = scalers[crop]["yield_std"]

    for scenario in scenarios:
        scenario_dir = os.path.join(ablation_base, crop, scenario)
        val_pkl = os.path.join(scenario_dir, "validation_outputs.pkl")
        test_pkl = os.path.join(scenario_dir, "test_outputs.pkl")

        # Skip scenarios with missing outputs
        if not os.path.exists(test_pkl) or not os.path.exists(val_pkl):
            print(f"⚠ Missing: {crop}/{scenario} — skipping")
            continue

        # Load validation and test outputs
        with open(val_pkl, "rb") as f:
            val_results = pickle.load(f)
        with open(test_pkl, "rb") as f:
            test_results = pickle.load(f)

        # Combine validation + test
        combined_preds = np.concatenate(
            [val_results["prediction"], test_results["prediction"]], axis=0
        )
        combined_targets = np.concatenate(
            [val_results["target"], test_results["target"]], axis=0
        )

        # De-normalize predictions and targets
        preds = combined_preds * target_std + target_mean  # (N, 3)
        targets = combined_targets * target_std + target_mean  # (N,)

        yhat = preds[:, 1]  # median (q0.5)
        ytrue = targets

        # Compute metrics
        r2 = r2_score(ytrue, yhat)
        rmse = math.sqrt(mean_squared_error(ytrue, yhat))
        mae = mean_absolute_error(ytrue, yhat)
        mape = mean_absolute_percentage_error(ytrue, yhat) * 100
        nrmse = rmse / ytrue.mean() * 100
        r, _ = pearsonr(ytrue, yhat)

        # Load param count from result JSON (if available)
        json_files = glob.glob(os.path.join(scenario_dir, "result_*.json"))
        total_params, trainable_params = None, None
        if json_files:
            with open(json_files[0], "r") as f:
                result_json = json.load(f)
            total_params = result_json.get("model_params", {}).get("total")
            trainable_params = result_json.get("model_params", {}).get("trainable")

        rows.append(
            {
                "Crop": crop_labels[crop],
                "Scenario": scenario_labels[scenario],
                "scenario_key": scenario,
                "r": round(r, 4),
                "R²": round(r2, 4),
                "RMSE": round(rmse, 4),
                "MAE": round(mae, 4),
                "NRMSE (%)": round(nrmse, 2),
                "MAPE (%)": round(mape, 2),
                "Params": trainable_params,
            }
        )

ablation_df = pd.DataFrame(rows)
print(
    f"✅ Loaded {len(ablation_df)} crop × scenario results (validation + test combined)"
)
ablation_df.head(10)

## Ablation metrics table


In [ ]:
# Styled multi-index table: Crop → Scenario
display_df = ablation_df.drop(columns=["scenario_key"]).set_index(["Crop", "Scenario"])

display(
    display_df.style.format(
        {
            "r": "{:.3f}",
            "R²": "{:.3f}",
            "RMSE": "{:.3f}",
            "MAE": "{:.3f}",
            "NRMSE (%)": "{:.2f}",
            "MAPE (%)": "{:.2f}",
            "Params": "{:,.0f}",
        }
    )
    .set_caption("Ablation study — Test set metrics per crop and model variant")
    .set_table_styles(
        [
            {
                "selector": "caption",
                "props": [
                    ("font-size", "13px"),
                    ("font-weight", "bold"),
                    ("text-align", "left"),
                ],
            },
            {"selector": "th", "props": [("text-align", "center")]},
            {"selector": "td", "props": [("text-align", "center")]},
        ]
    )
    .background_gradient(subset=["R²", "r"], cmap="RdYlGn", axis=0)
    .background_gradient(
        subset=["RMSE", "MAE", "NRMSE (%)", "MAPE (%)"], cmap="RdYlGn_r", axis=0
    )
)

## Bar plot — Ablation metrics across crops


In [ ]:
# ---- Grouped bar plots: one subplot per metric, bars = scenarios, groups = crops ----
scenario_order = [scenario_labels[s] for s in scenarios]
crop_order = [crop_labels[c] for c in crops]

scenario_colors = {
    "Full Model": "#4a86c8",
    "w/o VSN": "#e07941",
    "w/o Temporal Conv": "#2ea597",
    "w/o LSTM": "#da7596",
    "w/o Attention": "#8b5cf6",
    "w/o Static Enrichment": "#d4a843",
    "w/o Temporal Pooling": "#6b7280",
}

metrics_to_plot = ["R²", "RMSE", "NRMSE (%)", "MAPE (%)"]
metric_ylabels = {
    "R²": "$R^{2}$",
    "RMSE": "RMSE ($t\\ ha^{-1}$)",
    "NRMSE (%)": "NRMSE (%)",
    "MAPE (%)": "MAPE (%)",
}
# For R² higher is better; for the rest lower is better
higher_is_better = {"R²": True, "RMSE": False, "NRMSE (%)": False, "MAPE (%)": False}
panel_labels_bar = ["(a)", "(b)", "(c)", "(d)"]

n_metrics = len(metrics_to_plot)
n_crops = len(crop_order)
n_scenarios = len(scenario_order)

bar_width = 0.11
group_gap = 0.15
x_positions = np.arange(n_crops) * (n_scenarios * bar_width + group_gap)

fig, axes = plt.subplots(2, 2, figsize=(14, 9), dpi=300)
axes = axes.flatten()

for ax_idx, (ax, metric) in enumerate(zip(axes, metrics_to_plot)):
    # Identify the best value per crop for highlighting
    if higher_is_better[metric]:
        best_per_crop = ablation_df.groupby("Crop")[metric].max().to_dict()
    else:
        best_per_crop = ablation_df.groupby("Crop")[metric].min().to_dict()

    for i, scenario_name in enumerate(scenario_order):
        scenario_data = ablation_df[ablation_df["Scenario"] == scenario_name]
        vals = []
        for c in crop_order:
            match = scenario_data.loc[scenario_data["Crop"] == c, metric]
            vals.append(match.values[0] if len(match) > 0 else 0)

        offset = (i - (n_scenarios - 1) / 2) * bar_width
        bars = ax.bar(
            x_positions + offset,
            vals,
            width=bar_width,
            color=scenario_colors[scenario_name],
            edgecolor="k",
            linewidth=0.4,
            alpha=0.85,
            label=scenario_name if ax_idx == 0 else None,
        )

        # Value labels — bold if best
        for bar, val, crop in zip(bars, vals, crop_order):
            if val == 0:
                continue
            is_best = abs(val - best_per_crop.get(crop, float("inf"))) < 1e-6
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + ax.get_ylim()[1] * 0.008,
                f"{val:.2f}",
                ha="center",
                va="bottom",
                fontsize=6.5,
                fontweight="bold" if is_best else "normal",
                rotation=90,
                color="black",
            )

    ax.set_xticks(x_positions)
    ax.set_xticklabels(crop_order, fontsize=11)
    ax.set_ylabel(metric_ylabels[metric], fontsize=11)
    ax.tick_params(axis="y", labelsize=9)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.yaxis.grid(True, linestyle="--", alpha=0.5, zorder=0)
    ax.set_axisbelow(True)

    # Panel label
    ax.text(
        0.02,
        0.98,
        panel_labels_bar[ax_idx],
        transform=ax.transAxes,
        fontsize=14,
        fontweight="bold",
        va="top",
        ha="left",
    )

# Give room for rotated value labels
for ax in axes:
    ymin, ymax = ax.get_ylim()
    ax.set_ylim(ymin, ymax * 1.25)

# Shared legend below
legend_handles = [
    plt.Rectangle((0, 0), 1, 1, color=scenario_colors[s], ec="k", lw=0.4, label=s)
    for s in scenario_order
]
fig.legend(
    handles=legend_handles,
    loc="lower center",
    bbox_to_anchor=(0.5, -0.06),
    ncol=4,
    fontsize=10,
    frameon=False,
)

plt.tight_layout(h_pad=2, w_pad=2)
plt.show()

## Performance change (Δ) from full model when each component is removed


In [ ]:
# ---- Compute Δ metrics (ablated − full) for each crop ----
# Positive Δ for error metrics = degradation; Negative Δ for R² = degradation.
# We plot ΔR² (drop) and ΔRMSE (increase) to make interpretation intuitive.

ablation_only = [s for s in scenarios if s != "full_model"]
ablation_labels_only = [scenario_labels[s] for s in ablation_only]

delta_rows = []
for crop in crops:
    crop_label = crop_labels[crop]
    full = ablation_df[
        (ablation_df["Crop"] == crop_label)
        & (ablation_df["scenario_key"] == "full_model")
    ]
    if full.empty:
        continue

    for scenario in ablation_only:
        ablated = ablation_df[
            (ablation_df["Crop"] == crop_label)
            & (ablation_df["scenario_key"] == scenario)
        ]
        if ablated.empty:
            continue

        delta_rows.append(
            {
                "Crop": crop_label,
                "Removed Component": scenario_labels[scenario],
                "ΔR²": ablated["R²"].values[0] - full["R²"].values[0],
                "ΔRMSE": ablated["RMSE"].values[0] - full["RMSE"].values[0],
                "ΔNRMSE (pp)": ablated["NRMSE (%)"].values[0]
                - full["NRMSE (%)"].values[0],
                "ΔMAPE (pp)": ablated["MAPE (%)"].values[0]
                - full["MAPE (%)"].values[0],
            }
        )

delta_df = pd.DataFrame(delta_rows)

# Horizontal bar chart: one subplot per crop, bars = components
delta_metrics = ["ΔR²", "ΔRMSE"]
delta_ylabels = {
    "ΔR²": "$\\Delta R^{2}$",
    "ΔRMSE": "$\\Delta$RMSE ($t\\ ha^{-1}$)",
}

ablation_colors = {
    "w/o VSN": "#e07941",
    "w/o Temporal Conv": "#2ea597",
    "w/o LSTM": "#da7596",
    "w/o Attention": "#8b5cf6",
    "w/o Static Enrichment": "#d4a843",
    "w/o Pyramidal Pooling": "#6b7280",
}

n_crops_d = len(crop_order)
n_delta = len(delta_metrics)
panel_labels_delta = [f"({chr(ord('a') + i)})" for i in range(n_crops_d * n_delta)]

fig, axes = plt.subplots(
    n_crops_d, n_delta, figsize=(5 * n_delta, 3 * n_crops_d), dpi=300
)

for row, crop_label in enumerate(crop_order):
    crop_delta = delta_df[delta_df["Crop"] == crop_label].copy()
    if crop_delta.empty:
        continue

    for col, metric in enumerate(delta_metrics):
        ax = axes[row, col]
        panel_idx = row * n_delta + col

        # Sort by absolute impact (most impactful at the top)
        crop_delta_sorted = crop_delta.sort_values(metric, key=abs, ascending=True)

        components = crop_delta_sorted["Removed Component"].values
        values = crop_delta_sorted[metric].values

        # Color bars: red for degradation, green for improvement
        if metric == "ΔR²":
            colors = ["#d73027" if v < 0 else "#1a9850" for v in values]
        else:
            colors = ["#d73027" if v > 0 else "#1a9850" for v in values]

        bars = ax.barh(
            range(len(components)),
            values,
            color=colors,
            edgecolor="k",
            linewidth=0.5,
            alpha=0.85,
            height=0.6,
        )

        ax.set_yticks(range(len(components)))
        ax.set_yticklabels(components, fontsize=10)
        ax.axvline(0, color="k", linewidth=0.8, linestyle="-")
        ax.set_xlabel(delta_ylabels[metric], fontsize=11)
        ax.tick_params(axis="x", labelsize=9)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.xaxis.grid(True, linestyle="--", alpha=0.5, zorder=0)
        ax.set_axisbelow(True)

        # Value annotations on bars
        for bar, val in zip(bars, values):
            x_pos = val + (ax.get_xlim()[1] - ax.get_xlim()[0]) * 0.02 * np.sign(val)
            ax.text(
                x_pos,
                bar.get_y() + bar.get_height() / 2,
                f"{val:+.3f}",
                va="center",
                ha="left" if val >= 0 else "right",
                fontsize=8.5,
                fontweight="bold",
            )

        # Panel label
        ax.text(
            0.5,
            1.1,
            panel_labels_delta[panel_idx],
            transform=ax.transAxes,
            fontsize=13,
            fontweight="bold",
            va="top",
            ha="right",
        )

    # Row label (crop name)
    axes[row, 0].annotate(
        crop_label,
        xy=(0, 0.5),
        xytext=(-0.65, 0.5),
        xycoords="axes fraction",
        fontsize=12,
        fontweight="bold",
        va="center",
        ha="center",
        rotation=90,
    )

# Expand x limits for value annotations
for ax in axes.flatten():
    xmin, xmax = ax.get_xlim()
    margin = (xmax - xmin) * 0.2
    ax.set_xlim(xmin - margin, xmax + margin)

plt.tight_layout(w_pad=1, h_pad=1)
plt.subplots_adjust(left=0.15)

save_path = os.path.join(out_dir, "final", "ablation_delta_change.png")
os.makedirs(os.path.dirname(save_path), exist_ok=True)
fig.savefig(save_path, bbox_inches="tight", dpi=300)
print(f"Saved → {save_path}")
plt.show()

## Save results


In [ ]:
# Save the ablation metrics table
table_dir = (
    "/beegfs/halder/GITHUB/RESEARCH/crop-yield-forecasting-germany/output/tables"
)
os.makedirs(table_dir, exist_ok=True)

ablation_df.drop(columns=["scenario_key"]).to_csv(
    os.path.join(table_dir, "ablation_study_metrics.csv"), index=False
)
print(f"✅ Table saved to {os.path.join(table_dir, 'ablation_study_metrics.csv')}")

delta_df.to_csv(
    os.path.join(table_dir, "ablation_study_delta_metrics.csv"), index=False
)
print(
    f"✅ Delta table saved to {os.path.join(table_dir, 'ablation_study_delta_metrics.csv')}"
)